# SSE Streaming with LlamaBot

This notebook demonstrates how to use LlamaBot's Server-Sent Events (SSE) streaming capabilities for real-time chat applications.

In [ ]:
%load_ext autoreload
%autoreload 2

## Basic Streaming

`AsyncSimpleBot` (parallel to `SimpleBot`) exposes `stream_async()`, which yields content chunks asynchronously—ideal for SSE or WebSocket streaming.

In [ ]:
from llamabot import AsyncSimpleBot

bot = AsyncSimpleBot(
    system_prompt="You are a helpful assistant.",
    model_name="ollama/phi3",  # Use your preferred model
    stream_target="none",  # We'll handle streaming ourselves
)

In [ ]:
import asyncio

async def stream_example():
    chunks = []
    async for chunk in bot.stream_async("Tell me a short joke about programming."):
        chunks.append(chunk)
        print(chunk, end="", flush=True)  # Print as it streams
    print("\n\n---\n")
    print(f"Total chunks: {len(chunks)}")
    print(f"Full response: {''.join(chunks)}")

await stream_example()

## SSE Format Conversion

The `sse_stream()` utility converts bot streaming to SSE event format, ready for FastAPI's `EventSourceResponse`.

In [ ]:
from llamabot.sse import sse_stream

async def sse_example():
    events = []
    async for event in sse_stream(bot, ["What is Python?"], event_type="message", done_event="done"):
        events.append(event)
        print(f"Event: {event['event']}, Data: {repr(event['data'])}")

    print("\n---\n")
    print(f"Total events: {len(events)}")
    print(f"Last event type: {events[-1]['event']}")

await sse_example()

## FastAPI Integration

Here's how to use SSE streaming in a FastAPI endpoint:

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from sse_starlette.sse import EventSourceResponse

app = FastAPI()

class ChatRequest(BaseModel):
    messages: list[str]

@app.post("/chat")
async def chat(request: ChatRequest):
    """Stream chat responses using Server-Sent Events."""
    return EventSourceResponse(
        sse_stream(bot, request.messages, event_type="message", done_event="done")
    )

print("FastAPI endpoint defined!")
print("\nTo run this server:")
print("  uvicorn <module>:app --reload")
print("\nSee scripts/fastapi_sse_example.py for a complete example.")

## Error Handling

The `sse_stream()` function automatically handles errors and yields error events:

In [ ]:
from unittest.mock import MagicMock

# Create a bot that will raise an error
error_bot = MagicMock()

async def mock_stream_with_error(*args):
    yield "some"
    raise Exception("API error occurred")

error_bot.stream_async = mock_stream_with_error

async def error_example():
    events = []
    async for event in sse_stream(error_bot, ["test"]):
        events.append(event)
        print(f"Event: {event['event']}, Data: {event['data']}")

    print(f"\nTotal events: {len(events)}")
    print(f"Error event: {events[-1]}")

await error_example()

## `stream_async()` vs sync `__call__()`

Both paths can update **memory**, **SQLite logging**, and **spans** when streaming completes.

- **`SimpleBot.__call__()`**: Synchronous; uses `stream_target` (stdout, panel, api, none) for where incremental output goes.
- **`AsyncSimpleBot.stream_async()`**: Async token deltas via `litellm.acompletion`; intended for FastAPI/SSE. Does not use `stream_target` for those deltas.

Use **`SimpleBot()`** / **`__call__()`** for a single blocking `AIMessage` in scripts and notebooks. Use **`AsyncSimpleBot`** with **`stream_async()`** (or `await AsyncSimpleBot(...)`) when you need async iteration (SSE, WebSockets, etc.).

In [ ]:
print("\nComparison:")
print("\nSimpleBot().__call__() - Synchronous completion:")
print("  ✓ Returns AIMessage when done")
print("  ✓ Uses stream_target for console/panel/api")
print("\nAsyncSimpleBot.stream_async() - Async streaming:")
print("  ✓ Real-time text chunks (async iterator)")
print("  ✓ Memory / SQLite / spans on completion (same family as sync __call__)")
print("  ✓ SSE / WebSocket friendly")